In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src')
from pricing_engine import DynamicPricingEngine
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the pricing engine
engine = DynamicPricingEngine(
    '../src/models/demand_model.pkl',
    '../src/models/feature_columns.pkl'
)

# Load test data
df = pd.read_csv('../data/processed/hotel_bookings_processed.csv')
test_sample = df.sample(100, random_state=42)  # Random sample for testing

print("🚀 Testing Dynamic Pricing Engine")
print("=" * 50)

# =============================================================================
# 1. TEST DIFFERENT STRATEGIES
# =============================================================================

strategies = ['revenue_maximization', 'profit_maximization', 
              'market_penetration', 'premium_positioning']

strategy_results = {}

for strategy in strategies:
    results = engine.batch_pricing(
        test_sample, 
        strategy=strategy,
        constraints={'min_price': 40, 'max_price': 600, 'cost_per_night': 35}
    )
    
    strategy_results[strategy] = {
        'avg_price': results['recommended_price'].mean(),
        'total_revenue': results['expected_revenue'].sum(),
        'total_profit': results['expected_profit'].sum(),
        'avg_margin': results['profit_margin'].mean()
    }

print("\n📊 STRATEGY COMPARISON")
print("-" * 30)
comparison_df = pd.DataFrame(strategy_results).T
print(comparison_df.round(2))

# =============================================================================
# 2. PRICE SENSITIVITY ANALYSIS
# =============================================================================

print("\n📈 PRICE SENSITIVITY ANALYSIS")
print("-" * 30)

sensitivity_results = engine.simulate_pricing_impact(
    test_sample.head(20),
    price_changes=[-0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3],
    strategy='revenue_maximization'
)

print(sensitivity_results.round(2))

# =============================================================================
# 3. SEGMENT-BASED ANALYSIS
# =============================================================================

print("\n🏨 SEGMENT-BASED ANALYSIS")
print("-" * 30)

# FIX: Analyze different hotel types properly
hotel_analysis = []

for hotel_type in test_sample['hotel'].unique():
    hotel_data = test_sample[test_sample['hotel'] == hotel_type]
    
    if len(hotel_data) > 0:
        # Take up to 10 samples per hotel type
        sample_data = hotel_data.head(10) if len(hotel_data) >= 10 else hotel_data
        
        # Get pricing results for this hotel type
        pricing_results = engine.batch_pricing(sample_data)
        
        # Calculate summary metrics
        hotel_summary = {
            'hotel': hotel_type,
            'sample_size': len(sample_data),
            'recommended_price': pricing_results['recommended_price'].mean(),
            'expected_revenue': pricing_results['expected_revenue'].sum(),
            'profit_margin': pricing_results['profit_margin'].mean()
        }
        
        hotel_analysis.append(hotel_summary)

# Convert to DataFrame
segment_summary = pd.DataFrame(hotel_analysis).set_index('hotel').round(2)

print("Hotel Type Analysis:")
print(segment_summary)


# =============================================================================
# 4. VISUALIZATIONS
# =============================================================================

# Create comprehensive dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Strategy Comparison', 'Price Sensitivity', 
                   'Revenue vs Profit Trade-off', 'Pricing Distribution'],
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# Plot 1: Strategy Comparison
strategies_list = list(strategy_results.keys())
revenues = [strategy_results[s]['total_revenue'] for s in strategies_list]
profits = [strategy_results[s]['total_profit'] for s in strategies_list]

fig.add_trace(
    go.Bar(x=strategies_list, y=revenues, name='Revenue', 
           marker_color='lightblue'),
    row=1, col=1
)

# Plot 2: Price Sensitivity
fig.add_trace(
    go.Scatter(
        x=sensitivity_results['price_change'], 
        y=sensitivity_results['total_revenue'],
        mode='lines+markers', 
        name='Revenue Impact',
        line=dict(color='blue')
    ),
    row=1, col=2
)

# Plot 3: Revenue vs Profit Trade-off
fig.add_trace(
    go.Scatter(
        x=revenues, y=profits,
        mode='markers+text',
        text=strategies_list,
        textposition="top center",
        name='Strategy Trade-offs',
        marker=dict(size=12, color='red')
    ),
    row=2, col=1
)

# Plot 4: Price Distribution
sample_results = engine.batch_pricing(test_sample.head(50))
fig.add_trace(
    go.Histogram(
        x=sample_results['recommended_price'],
        nbinsx=20,
        name='Price Distribution',
        marker_color='green',
        opacity=0.7
    ),
    row=2, col=2
)

fig.update_layout(
    height=800, 
    showlegend=True,
    title_text="Dynamic Pricing Engine Performance Dashboard"
)

fig.show()

# =============================================================================
# 5. SAVE RESULTS
# =============================================================================

# Save all results for further analysis
results_summary = {
    'strategy_comparison': comparison_df,
    'price_sensitivity': sensitivity_results,
    'test_sample_pricing': sample_results
}

# Save to files
comparison_df.to_csv('../data/processed/strategy_comparison.csv')
sensitivity_results.to_csv('../data/processed/price_sensitivity_analysis.csv')
sample_results.to_csv('../data/processed/sample_pricing_results.csv')

print("\n✅ All results saved to data/processed/")
print("\n🎯 Key Insights:")
print(f"1. Best strategy for revenue: {comparison_df['total_revenue'].idxmax()}")
print(f"2. Best strategy for profit: {comparison_df['total_profit'].idxmax()}")
print(f"3. Most price-sensitive change: {sensitivity_results.loc[sensitivity_results['total_revenue'].idxmax(), 'price_change']}")
print(f"4. Average recommended price: ${sample_results['recommended_price'].mean():.2f}")



INFO:pricing_engine:Dynamic Pricing Engine initialized successfully


🚀 Testing Dynamic Pricing Engine

📊 STRATEGY COMPARISON
------------------------------
                      avg_price  total_revenue  total_profit  avg_margin
revenue_maximization     152.34      152336.70      99018.81       77.18
profit_maximization      124.52      124521.51      80939.01       66.71
market_penetration        87.37       87365.05      56787.29       53.79
premium_positioning      128.85      128851.39      83753.38       67.96

📈 PRICE SENSITIVITY ANALYSIS
------------------------------
   price_change  avg_price  total_demand  total_revenue  total_profit  \
0          -0.3     140.15       1208.78      199214.99     150864.44   
1          -0.2     140.15       1208.78      199214.99     150864.44   
2          -0.1     140.15       1208.78      199214.99     150864.44   
3           0.0     140.15       1208.78      199214.99     150864.44   
4           0.1     140.15       1208.78      199214.99     150864.44   
5           0.2     140.15       1208.78      199


✅ All results saved to data/processed/

🎯 Key Insights:
1. Best strategy for revenue: revenue_maximization
2. Best strategy for profit: revenue_maximization
3. Most price-sensitive change: -0.3
4. Average recommended price: $153.27

🚀 Phase 4 Complete! Ready for next phase: Interactive Demo
